# StegaGuard Phase 5: Kaggle classical steganalysis

This notebook reads BOSSbase from Kaggle input storage, prepares 256×256 grayscale covers, trains the classical detectors, and writes models and metrics under `/kaggle/working`. Nothing is downloaded to your laptop.

## Before running
1. In Kaggle, create a notebook and use **Add Input** to attach the raw [BOSSbase 1.01 dataset](https://www.kaggle.com/datasets/lunan0/bossbase-1-01). Review its dataset terms and citation requirements.
2. The notebook also needs the project source. From the repository root on Windows, create a small source archive with `Compress-Archive -Path backend -DestinationPath StegaGuard-code.zip`. Upload that ZIP as a Kaggle Dataset named `stegaguard-source`, then attach it with **Add Input**. Do not include your BOSSbase files in this archive.
3. Add a Kaggle Secret named `STEGAGUARD_TRAINING_PASSPHRASE` (at least 8 characters). It is used only to generate keyed-mode training variants; it is not printed or saved in model metadata.
4. If Kaggle gives either input a different folder name, edit the two paths in the setup cell.

The complete run covers payloads 0.05, 0.1, 0.2, 0.4, and 1.0 bpp; both LSB methods and both modes. This is CPU-heavy and may take a long time. Outputs are experimental results only when this full notebook run completes; capped or synthetic smoke runs are not benchmark results.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

BOSSBASE_ROOT = Path("/kaggle/input/bossbase-1-01")
SOURCE_INPUT = Path("/kaggle/input/stegaguard-source")
WORK_ROOT = Path("/kaggle/working/stegaguard-phase5")
SOURCE_WORK = WORK_ROOT / "source"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if not BOSSBASE_ROOT.is_dir():
    print("Available Kaggle inputs:", [path.name for path in Path("/kaggle/input").iterdir()])
    raise FileNotFoundError("Edit BOSSBASE_ROOT to the attached raw BOSSbase dataset folder.")
if not SOURCE_INPUT.is_dir():
    raise FileNotFoundError(
        "Attach the stegaguard-source Kaggle Dataset and edit SOURCE_INPUT if needed."
    )

archives = sorted(SOURCE_INPUT.glob("*.zip"))
if archives:
    SOURCE_WORK.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archives[0]), str(SOURCE_WORK))
    source_root = SOURCE_WORK
else:
    source_root = SOURCE_INPUT
prepare_script = next(source_root.rglob("prepare_dataset.py"), None)
if prepare_script is None:
    raise FileNotFoundError("The source input must contain backend/scripts/prepare_dataset.py.")
BACKEND = prepare_script.parent.parent
print("Backend source:", BACKEND)
print("Raw dataset input:", BOSSBASE_ROOT)
print("Kaggle working output:", WORK_ROOT)

In [ ]:
import joblib
import numpy
import PIL
import scipy
import sklearn

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Pillow:", PIL.__version__)
print("joblib:", joblib.__version__)

## Prepare the covers

This step reads the attached raw images directly and writes resized PNGs plus deterministic cover-level splits into Kaggle's writable working directory.

In [ ]:
PREPARED = WORK_ROOT / "prepared"
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.prepare_dataset",
        "--src",
        str(BOSSBASE_ROOT),
        "--out",
        str(PREPARED),
        "--size",
        "256",
        "--seed",
        "1337",
        "--origin",
        "BOSSbase 1.01 from Kaggle (lunan0/bossbase-1-01)",
    ],
    cwd=BACKEND,
    check=True,
)

## Train and evaluate

Training uses only the training covers. Decision thresholds are selected on validation variants, then RF, HGB, and the clearly labelled untrained statistical fallback are evaluated per method/mode/payload on cover-disjoint validation and test splits.

In [ ]:
from kaggle_secrets import UserSecretsClient

training_passphrase = UserSecretsClient().get_secret("STEGAGUARD_TRAINING_PASSPHRASE")
if len(training_passphrase) < 8:
    raise ValueError("The Kaggle training secret must be at least 8 characters.")
os.environ["STEGAGUARD_TRAINING_PASSPHRASE"] = training_passphrase
del training_passphrase

MODELS = WORK_ROOT / "models"
RESULTS = WORK_ROOT / "results" / "classical_metrics.json"
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.train_classical",
        "--dataset",
        str(PREPARED),
        "--models-out",
        str(MODELS),
        "--results-out",
        str(RESULTS),
        "--seed",
        "1337",
        "--payloads",
        "0.05,0.1,0.2,0.4,1.0",
        "--methods",
        "lsb_replace,lsb_match",
        "--modes",
        "sequential,keyed",
        "--forest-trees",
        "300",
    ],
    cwd=BACKEND,
    check=True,
)

## Download the trained artifacts

The ZIP contains only model weights, JSON sidecars, and test metrics, not the BOSSbase images. Save a notebook version to retain the files in Kaggle output, or download the ZIP to your laptop.

In [ ]:
from zipfile import ZipFile

artifact_zip = WORK_ROOT / "stegaguard-classical-artifacts.zip"
with ZipFile(artifact_zip, "w") as archive:
    for file_path in sorted(MODELS.glob("*")):
        if file_path.is_file():
            archive.write(file_path, arcname=f"models/{file_path.name}")
    archive.write(RESULTS, arcname="results/classical_metrics.json")
print("Artifacts:", artifact_zip)
print("Metrics:", RESULTS)